# Lesson 1.9: EDA Advanced — Data Wrangling & Analysis

Lesson 1.8 asked *"can I trust this data?"*. This lesson asks the next question:
**what is the pattern, and what should we do about it?**

Clean rows on their own answer nothing. You have to put time on the index, join in the tables that
give the rows meaning, reshape them, and group them. That is the whole job here.

**Structure — the four learning outcomes, in order:**
* **Part 1: Time Series** — *parse* dates, then resample and roll them.
* **Part 2: Data Integration** — *merge* tables, and convert wide ↔ long.
* **Part 3: Aggregation & Reporting** — *aggregate* with `groupby`, `pivot_table`, `crosstab`.
* **Part 4: From Table to Decision** — *apply* all of it to answer the owner's actual question.

**How to read the code cells:** read the `# 👉` comment above each line before you run the cell. The comment says
what the line does in plain English; the output shows you it happened.


> **🧭 Today's flow — 180 minutes.** One business problem, four learning outcomes, in order:
>
> | | Section | Learning outcome | Time |
> |---|---|---|---|
> | — | Setup + why this matters | | 5 min |
> | **Part 1** | Time Series | **Parse** dates; `resample`, `rolling`, `shift` | 45 min |
> | ☕ | *Break* | | 10 min |
> | **Part 2** | Data Integration | **Merge** tables; `melt` / `pivot` (wide ↔ long) | 45 min |
> | ☕ | *Break* | | 10 min |
> | **Part 3** | Aggregation & Reporting | **Aggregate**: `groupby`, `pivot_table`, `crosstab` | 45 min |
> | **Part 4** | From Table to Decision | **Apply** split-apply-combine to the real question | 20 min |
>
> **The spine:** one business problem — *The Daily Grind*, a four-outlet café chain — and one main
> file, `data/daily_sales.csv`, from start to finish. Small hand-built tables appear alongside as
> *drills*: they isolate one method so you can see exactly what it does.
>
> Each of Parts 1–3 ends with a **🛠️ Group Exercise**. Deep dives live in `reference.md`.


### The four beats of every summary

Lesson 1.8 gave you four beats for every fix: **find it → decide → apply → verify.**
Summarising has its own four, and every table we build today follows them:

| Beat | Ask yourself | |
|---|---|---|
| **1. Question** | What decision does this number serve? | *Renew the Marina Bay lease — yes or no?* |
| **2. Grain** | One row per **what**? | *One row per outlet, per month* |
| **3. Aggregation** | Sum, mean or count — and **why that one**? | *Sum for totals, mean for efficiency* |
| **4. Check** | Does the total still tie back? | *Grouped total == ungrouped total* |

Beat 4 is the one everyone skips. In Part 2 it catches a join that silently deletes $61,310.


### Setup

Import the libraries, then load the file we will use all session.


In [ ]:
# 👉 Two toolkits. `pd` and `np` are just short nicknames, so we can type `pd.something`
#    instead of `pandas.something`. Run this cell first, every session.
import pandas as pd
import numpy as np

# 👉 Housekeeping only. Pandas renames a few option strings between versions and shouts
#    about it; this keeps those notices out of our output. Nothing to learn here.
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
# 👉 The spine. One row per outlet, per day, per part of the day (Morning/Midday/Evening).
#    18 months of trading for a four-outlet café chain, plus a pop-up kiosk.
#    `parse_dates=["date"]` tells pandas: this column is not text, it is a date. More on that in 1.1.
sales = pd.read_csv("../data/daily_sales.csv", parse_dates=["date"])

sales.head()


In [ ]:
# 👉 The 1.8 habit still applies: look before you leap. Shape, types, holes.
print("rows, columns:", sales.shape)
sales.info()


### 🎬 Why this matters — the flat line that hides everything

**The situation.** *The Daily Grind* runs four cafés in Singapore. Revenue has been flat for two
quarters. The Marina Bay lease is up for renewal this month, rent is $9,600, and the owner has to
sign or walk away. She sends you the sales export and asks one question: **what is going on?**

Run the next two cells. The first is the number she already has. The second is the same number,
split by outlet.

> Do not worry about how these two lines work yet — that is Part 1 and Part 3. Just read the output.


In [ ]:
# 👉 Total revenue per quarter for the whole chain -- the headline the owner already has.
#    (`.to_period("Q")` labels each date with its calendar quarter.)
chain_by_quarter = sales.groupby(sales["date"].dt.to_period("Q"))["revenue_sgd"].sum().round(0)

chain_by_quarter


In [ ]:
# 👉 The same revenue, but one column per outlet. Same data. Same period. Different question.
by_outlet = sales.pivot_table(
    index=sales["date"].dt.to_period("Q"),   # down the side: quarter
    columns="outlet_id",                     # across the top: outlet
    values="revenue_sgd",                    # the number in the middle
    aggfunc="sum",                           # how to squash it: add it up
).round(0)

by_outlet


**Read the second table.** OUT-03 falls from about \$138k a quarter to about \$100k. OUT-04 climbs
from about \$96k to about \$131k. One is dying, one is growing, and they move by almost the same
amount — so the chain total barely twitches.

The flat line was never the story. It was **two opposite stories cancelling out**.

No amount of cleaning would have found this. Cleaning gives you rows you can trust; only grouping
turns them into an answer. Three more things you cannot see yet, and will by the end of the session:

1. OUT-03's fall is not a slope. It is a **step**, on one specific week. (Part 1)
2. There is a fifth outlet in this file that does not exist in the outlet list. (Part 2)
3. OUT-03 is still staffed for the revenue it used to make. (Part 3)

Write those three down. We will tick them off.


---

## Part 1: Time Series — Making Time a First-Class Column

**Learning outcome 1:** *Parse and manipulate datetime data to perform time-based resampling and
rolling window calculations.*

**Goal:** the owner's question is about *change over time*. Before pandas can answer anything about
time, it has to know that a date is a date — not a piece of text that happens to look like one.

⏱️ ~45 min including Group Exercise 1


### 1.1: A date is a type, not a string

This is the whole idea of the section, so it is worth ten seconds of proof.


In [ ]:
# 👉 Load the same file again, but WITHOUT telling pandas that `date` is a date.
#    `object` in the output below means "text". Pandas has no idea these are dates.
raw = pd.read_csv("../data/daily_sales.csv")

raw["date"].dtype


In [ ]:
# 👉 Here is why that matters. As text, "2024-10-02" sorts before "2024-9-30" -- alphabetically,
#    "1" comes before "9". Dates as text sort like words, not like time.
text_dates = pd.Series(["2024-10-02", "2024-9-30", "2024-11-01"])

text_dates.sort_values()


In [ ]:
# 👉 `pd.to_datetime` converts text into real timestamps. Now sorting means what you expect.
real_dates = pd.to_datetime(text_dates)

real_dates.sort_values()


In [ ]:
# 👉 Not every system writes dates the American way. "01/06/2025" is 1 June in Singapore and
#    6 January in the US -- and pandas cannot know which you meant. So tell it.
#    `format=` spells the layout out: %d day, %m month, %Y four-digit year.
uk_style = pd.Series(["01/06/2025", "02/06/2025", "03/06/2025"])

pd.to_datetime(uk_style, format="%d/%m/%Y")


In [ ]:
# 👉 `dayfirst=True` is the shorter way to say the same thing when the layout is consistent.
pd.to_datetime(uk_style, dayfirst=True)


> **This is the single most common silent bug in date handling.** Without `format=` or
> `dayfirst=True`, pandas guesses from the first value it can parse and then applies that guess to
> the whole column. Days 1–12 of a month parse "successfully" under the wrong reading, so
> `"03/06/2025"` becomes 6 March and no error is raised. Your report is then wrong by three months
> and looks fine.
>
> If your dates genuinely change format row to row, pandas 2.0+ has `format="mixed"`. In the `pds`
> environment (pandas 1.5) you would clean the column first — one format at a time.


In [ ]:
# 👉 Once a column is a real date, the `.dt` accessor unlocks date questions --
#    exactly like `.str` unlocked text questions in 1.8.
print("first day:", sales["date"].min().date())
print("last day: ", sales["date"].max().date())
print("days covered:", (sales["date"].max() - sales["date"].min()).days + 1)

sales["date"].dt.day_name().head(3)


In [ ]:
# 👉 Pull date parts out into their own columns so we can group by them later.
sales["month"] = sales["date"].dt.to_period("M")   # 2024-01, 2024-02, ...
sales["weekday"] = sales["date"].dt.day_name()     # Monday, Tuesday, ...
sales["is_weekend"] = sales["date"].dt.dayofweek >= 5   # Saturday=5, Sunday=6

sales[["date", "month", "weekday", "is_weekend"]].head()


> **Beat 2 in action.** `sales` has one row per outlet **per day per daypart** — three rows per
> outlet per day. Any total you take without saying which grain you want will quietly mix them.


### 1.2: The DatetimeIndex — put time on the index

A `DatetimeIndex` is a date column promoted to be the row label. It is what unlocks `.resample()`,
`.rolling()`, and slicing by `"2025-06"` instead of writing a filter.


In [ ]:
# 👉 One number per day for the whole chain: add up every outlet and every daypart on that date.
#    The result is a Series whose INDEX is the date -- that is a DatetimeIndex.
chain_daily = sales.groupby("date")["revenue_sgd"].sum()

print(type(chain_daily.index).__name__)
chain_daily.head()


In [ ]:
# 👉 With dates on the index you can slice with plain strings. This is one month:
chain_daily["2025-06"].head()


In [ ]:
# 👉 ...and this is a range. Both ends are INCLUDED with `.loc` on dates (unlike normal Python).
chain_daily.loc["2024-11-01":"2024-11-05"]


In [ ]:
# 👉 Drill on a hand-built series, so the mechanics are visible.
#    `date_range` makes a run of dates; here: 6 days starting 1 Jan.
idx = pd.date_range("2024-01-01", periods=6, freq="D")
drill = pd.Series([10, 12, 9, 15, 11, 14], index=idx)

drill


In [ ]:
# 👉 Real data has gaps. Drop two days, then `reindex` back onto the full calendar:
#    the missing days come back as NaN instead of silently disappearing.
gappy = drill.drop([pd.Timestamp("2024-01-03"), pd.Timestamp("2024-01-04")])

gappy.reindex(idx)


> **Why the gap matters.** A missing day and a zero-revenue day look identical in a chart, and mean
> opposite things: "closed for a public holiday" vs "open and sold nothing". `reindex` makes the
> difference visible before you average anything.


### 1.3: `resample` — change the grain of time

`resample` is `groupby` for dates. You give it a frequency; it buckets the rows and aggregates
each bucket. **Beat 3 applies:** the frequency is the grain, the aggregation is your choice, and
choosing wrong gives a confident wrong answer.

| Alias | Bucket |
|---|---|
| `D` | calendar day |
| `W` | week (ending Sunday by default) |
| `M` | month end |
| `Q` | quarter end |
| `A` / `Y` | year end |


In [ ]:
# 👉 Daily -> monthly, adding up each month. This is the owner's headline number.
monthly_chain = chain_daily.resample("M").sum().round(0)

monthly_chain.tail(8)


In [ ]:
# 👉 The same resample with `.mean()` answers a DIFFERENT question: an average TRADING DAY.
#    Sum is distorted by month length (February is short); mean is not. Neither is "correct" --
#    they answer different questions. That is beat 3.
compare = pd.DataFrame({
    "total_revenue": chain_daily.resample("M").sum().round(0),
    "avg_day": chain_daily.resample("M").mean().round(0),
    "trading_days": chain_daily.resample("M").size(),
})

compare.tail(6)


> **Look at February 2025.** The total drops and the average day barely moves. The "drop" was 28
> days versus 31, plus Chinese New Year — not a business problem. A manager shown only the totals
> column would go looking for a cause that does not exist.


In [ ]:
# 👉 Resampling upwards (finer) instead of downwards creates rows that did not exist, so you must
#    say how to fill them. `.asfreq()` leaves NaN; `.ffill()` carries the last value forward.
weekly = chain_daily.resample("W").sum()

weekly.resample("D").asfreq().head(4)


### 1.4: `rolling` — smooth the noise to see the shape

Daily café revenue swings wildly between weekdays and weekends. A rolling (moving) average replaces
each day with the average of it and the days before it, which strips out the weekly rhythm and
leaves the trend.


In [ ]:
# 👉 Marina Bay only. One number per day for OUT-03.
marina = sales[sales["outlet_id"] == "OUT-03"].groupby("date")["revenue_sgd"].sum()

marina.loc["2024-10-28":"2024-11-03"].round(0)


In [ ]:
# 👉 A 7-day rolling mean: each value is the average of that day and the 6 before it.
#    7 days = exactly one week, so it cancels the weekday/weekend cycle.
#    The first 6 values are NaN -- there is nothing behind them to average.
marina_7d = marina.rolling(window=7).mean()

marina_7d.head(9).round(0)


In [ ]:
# 👉 Now put the raw and the smoothed side by side across the first week of November 2024.
#    Ignore the daily zig-zag and read the `smooth_7d` column downwards.
pd.DataFrame({
    "raw": marina.round(0),
    "smooth_7d": marina_7d.round(0),
}).loc["2024-10-28":"2024-11-14"]


In [ ]:
# 👉 A 28-day window smooths harder. Compare the monthly averages either side of 4 Nov 2024:
before = marina.loc["2024-09-01":"2024-11-03"].mean()
after = marina.loc["2024-11-04":"2025-01-31"].mean()

print(f"average day before 4 Nov 2024: ${before:,.0f}")
print(f"average day after  4 Nov 2024: ${after:,.0f}")
print(f"change: {(after / before - 1) * 100:,.1f}%")


> **Tick off finding #1.** That is not a slope, it is a **step** — one week, then a new normal.
> Slopes and steps have different causes. A slope says "we are slowly losing our regulars"; a step
> says "something happened on that date". (A competitor opened next door on 4 November 2024.)
>
> A rolling average is the cheapest tool you own for telling those two apart.


In [ ]:
# 👉 `.shift()` moves the values down by n rows, which lets you compare a period with the one
#    before it. `.pct_change()` is the same idea, packaged: (this - previous) / previous.
mom = pd.DataFrame({
    "revenue": monthly_chain,
    "prev_month": monthly_chain.shift(1),
    "change_pct": (monthly_chain.pct_change() * 100).round(1),
})

mom.tail(6)


> **Do not trust a percentage until you know what is in the denominator.** March 2025 is up 25%
> and June is down 13% — and neither is about the cafés. A pop-up kiosk traded from March to May
> and is in this total. Month-on-month change is the most over-quoted number in business reporting
> precisely because it moves for reasons like that.


### 1.5: The payoff — resample per outlet

One `resample` on the chain hid the story. The same resample, done per outlet, reveals it.


In [ ]:
# 👉 Reading it inside out:
#    (1) pivot so each outlet is a column and each date is a row,
#    (2) resample those rows to month-end totals.
#    `pivot_table` with `aggfunc="sum"` collapses the three dayparts per day into one number.
by_outlet_daily = sales.pivot_table(
    index="date", columns="outlet_id", values="revenue_sgd", aggfunc="sum"
)

monthly_by_outlet = by_outlet_daily.resample("M").sum()

monthly_by_outlet.tail(6).round(0)


In [ ]:
# 👉 Beat 4: the check. Do the per-outlet monthly totals still add up to the chain total?
#    If this prints False, we lost rows somewhere and every number above is suspect.
#    `np.isclose` compares with a tiny tolerance -- decimals of a cent should not fail a check.
grouped_total = monthly_by_outlet.sum().sum()
raw_total = sales["revenue_sgd"].sum()

print("per-outlet total ties back to the raw total:", np.isclose(grouped_total, raw_total))
print(f"grouped: ${grouped_total:,.2f}")
print(f"raw:     ${raw_total:,.2f}")


> **Compare exactly the thing you care about.** Note that we checked the *unrounded* totals. Had we
> rounded each monthly figure to the nearest dollar first and then added them up, the check would
> have failed by a few dollars — 75 separate roundings — and sent us hunting for a data-loss bug that
> did not exist. Round for display, never before a comparison.


### 🛠️ Group Exercise 1 — Time (10 min)

The scaffold fades: (a) is done for you, (b) is half-written, (c) and (d) are yours.

> **(a) Worked for you** — the cell below shows revenue by day of the week for Raffles Place.
> Read it, run it, and say out loud what kind of café this is.
>
> **(b) Fill in the blanks** — the same weekday profile for Tampines Mall (`OUT-02`):
> ```python
> t = sales[sales["outlet_id"] == "______"]
> t.groupby("weekday")["revenue_sgd"].____().round(0)
> ```
> *Expected:* Saturday and Sunday are the two **biggest** days — the opposite of Raffles Place.
>
> **(c) From scratch** — resample Holland Village (`OUT-04`) to monthly totals and find its best
> month. *Hint:* build the daily series with `groupby("date")`, then `.resample("M").sum()`,
> then `.idxmax()`. *Expected:* a month in the first half of 2025.
>
> **(d) Explain, no code** — `chain_daily.rolling(7).mean()` has 6 NaN values at the start, and
> `chain_daily.resample("M").mean()` has none. Why? Which would you show a manager, and why?


**(a) Worked example** — the weekday profile of a CBD café:


In [ ]:
# 👉 Filter to one outlet, group the rows by weekday name, add up revenue in each group.
#    `.reindex([...])` puts the days back in calendar order (groupby sorts alphabetically).
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

raffles = sales[sales["outlet_id"] == "OUT-01"]
raffles.groupby("weekday")["revenue_sgd"].sum().reindex(day_order).round(0)


---

# ☕ Break — 10 minutes

**Where we are:** time is now a real column, and you can change its grain (`resample`) and smooth
it (`rolling`). You found the step at Marina Bay.

**Next up:** Part 2 — the sales file only knows outlet *codes*. To say anything about rent, region
or staffing, we have to join in the other tables.


## Part 2: Data Integration — Joining and Reshaping

**Learning outcome 2:** *Merge multiple DataFrames using SQL-style joins and convert between wide
and long data formats.*

**Goal:** `daily_sales.csv` contains `OUT-03`, not "Marina Bay", and it knows nothing about rent,
seats or staff hours. Those live in other files. Joining is how a row gets its meaning.

⏱️ ~45 min including Group Exercise 2


### 2.1: `merge` — the lookup table

A merge lines up two tables on a shared **key** column. Here the key is `outlet_id`.


In [ ]:
# 👉 The lookup table: one row per outlet, with the attributes the sales file lacks.
outlets = pd.read_csv("../data/outlets.csv", parse_dates=["opened_date"])

outlets


In [ ]:
# 👉 Beat 2 first: choose the grain BEFORE joining. One row per outlet per month.
#    `.reset_index()` turns the grouped index back into ordinary columns so we can merge on them.
monthly = (
    sales.groupby(["outlet_id", "month"])["revenue_sgd"].sum().round(2).reset_index()
)

monthly.head()


In [ ]:
# 👉 The merge. `on="outlet_id"` is the key; `how="left"` means "keep every row on the left,
#    and attach matching columns from the right where they exist".
monthly_named = monthly.merge(outlets, on="outlet_id", how="left")

monthly_named.head(3)


#### The four `how`s, and why the choice is not cosmetic

Our two files disagree about which outlets exist, on purpose:

- `daily_sales.csv` has **OUT-05** — a pop-up kiosk that never made it into the outlet list.
- `outlets.csv` has **OUT-06** (Sentosa Cove) — signed but not yet open, so it has no sales.

Watch what each join does to those two.


In [ ]:
# 👉 The same merge four ways. `.shape[0]` is the row count; the revenue total is beat 4.
for how in ["inner", "left", "right", "outer"]:
    m = monthly.merge(outlets, on="outlet_id", how=how)
    print(
        f"{how:>6}: {m.shape[0]:>3} rows | "
        f"revenue ${m['revenue_sgd'].sum():>12,.0f} | "
        f"outlets seen: {sorted(m['outlet_id'].unique())}"
    )

print(f"\n  raw sales total: ${sales['revenue_sgd'].sum():,.0f}")


> **Tick off finding #2.** The inner join is short by **\$61,310** — the pop-up kiosk's entire
> takings — and it does not warn you. It just quietly returns a smaller number that looks fine.
>
> | `how` | Keeps | Use when |
> |---|---|---|
> | `inner` | only keys in **both** | you need complete attributes on every row |
> | `left` | all of the **left** | the left table is your spine and must not shrink — **the default choice for analysis** |
> | `right` | all of the **right** | rarely; usually clearer written as a `left` the other way round |
> | `outer` | **everything** | reconciling two lists, when the mismatches *are* the finding |
>
> Rule of thumb: use `left` and then check for nulls. Reach for `inner` only when you can say why
> losing rows is correct.


In [ ]:
# 👉 `indicator=True` adds a `_merge` column saying where each row came from.
#    This is the fastest way to see a mismatch instead of guessing at it.
audit = monthly.merge(outlets, on="outlet_id", how="outer", indicator=True)

audit["_merge"].value_counts()


In [ ]:
# 👉 Which rows are the problem ones? Anything not "both".
audit.loc[audit["_merge"] != "both", ["outlet_id", "month", "revenue_sgd", "outlet_name", "_merge"]].head()


In [ ]:
# 👉 `validate=` makes pandas check the shape of the relationship and raise if it is wrong.
#    "many_to_one": many sales rows, one outlet row. If `outlets` ever gained a duplicate
#    outlet_id, this line would fail loudly instead of silently doubling your revenue.
monthly.merge(outlets, on="outlet_id", how="left", validate="many_to_one").shape


### 2.2: Merging on two keys — the roster

Sometimes one column is not enough to identify a row. The roster is one row per outlet **per week**,
so the key is the pair `(outlet_id, week_start)`.


In [ ]:
# 👉 Weekly staffing per outlet.
roster = pd.read_csv("../data/roster.csv", parse_dates=["week_start"])

roster.head(3)


In [ ]:
# 👉 To join, our sales must be at the same grain: one row per outlet per week, weeks starting
#    Monday. `pd.Grouper` is how you resample INSIDE a groupby of something else.
#    `label="left"` labels each week with its first day, matching the roster's `week_start`.
weekly_sales = (
    sales.groupby(["outlet_id", pd.Grouper(key="date", freq="W-MON", label="left")])["revenue_sgd"]
    .sum()
    .reset_index()
    .rename(columns={"date": "week_start"})
)

weekly_sales.head(3)


In [ ]:
# 👉 Merge on BOTH keys, as a list. Get either key wrong and you get a silent Cartesian mess --
#    which is exactly what `validate="one_to_one"` is there to prevent.
staffed = weekly_sales.merge(roster, on=["outlet_id", "week_start"], how="inner", validate="one_to_one")

# 👉 The efficiency question the owner actually cares about: dollars earned per staff hour paid.
staffed["rev_per_staff_hour"] = (staffed["revenue_sgd"] / staffed["staff_hours"]).round(2)

staffed.head(3)


> **Watch the dtypes when you merge.** `week_start` had to be a real date on *both* sides. Merging
> a text `"2024-01-01"` against a `Timestamp("2024-01-01")` matches nothing, and pandas reports
> zero matches rather than an error. If a merge returns suspiciously few rows, check `.dtypes` first.


### 2.3: Wide → long with `melt`

The targets file is laid out the way a manager types it into Excel: one row per outlet, **one
column per month**. Convenient for reading, useless for joining — "month" is not a column, it is
a set of headers.


In [ ]:
# 👉 Look at the shape of the problem first.
targets_wide = pd.read_csv("../data/targets_wide.csv")

targets_wide.iloc[:, :6]


In [ ]:
# 👉 `melt` unpivots. `id_vars` are the columns to KEEP as they are; everything else gets folded
#    down into two new columns: one holding the old header, one holding the value.
targets = targets_wide.melt(
    id_vars="outlet_id",        # keep this as a column
    var_name="month",           # the old column headers land here
    value_name="target_sgd",    # the numbers land here
)

print(f"{targets_wide.shape} wide  ->  {targets.shape} long")
targets.head()


In [ ]:
# 👉 Now `month` is a real column, so it can be a merge key. Both sides must be the same type,
#    so convert our Period month to text to match the target sheet's "2024-01" strings.
monthly["month_str"] = monthly["month"].astype(str)

performance = monthly.merge(targets, left_on=["outlet_id", "month_str"], right_on=["outlet_id", "month"], how="left")
performance["variance_pct"] = ((performance["revenue_sgd"] / performance["target_sgd"] - 1) * 100).round(1)

performance[["outlet_id", "month_str", "revenue_sgd", "target_sgd", "variance_pct"]].tail(8)


> **The variance column is the point.** Every month of it was invisible while the targets sat in a
> wide sheet. Reshaping is not tidiness for its own sake — it is what makes a comparison possible.
>
> Notice the last three rows: the pop-up kiosk has a `NaN` target, because nobody set it one. That
> is the honest answer. If you had used `.fillna(0)` here, the kiosk would show a variance of
> "+∞% above target" — a number that is arithmetically fine and completely false.


### 2.4: Long → wide with `pivot`

`pivot` is `melt` run backwards: it takes a column of labels and spreads it across the top.
Long format is for computers; wide format is for people. Reshape at the last minute, for reading.


In [ ]:
# 👉 One row per month, one column per outlet. `pivot` needs three things: what goes down the
#    side (index), what goes across the top (columns), and what fills the middle (values).
wide_view = monthly.pivot(index="month_str", columns="outlet_id", values="revenue_sgd").round(0)

wide_view.tail(6)


In [ ]:
# 👉 `pivot` fails if a single (index, columns) cell would hold more than one value -- it has
#    no instruction for what to do with the second one. Here, two rows land in the same cell:
clash = pd.DataFrame({
    "month": ["2025-06", "2025-06"],
    "outlet_id": ["OUT-01", "OUT-01"],
    "revenue_sgd": [100, 200],
})

try:
    clash.pivot(index="month", columns="outlet_id", values="revenue_sgd")
except ValueError as e:
    print("ValueError:", e)


In [ ]:
# 👉 `pivot_table` is the same operation PLUS an aggregation, so duplicates are fine --
#    you tell it how to combine them. That is the only real difference between the two.
clash.pivot_table(index="month", columns="outlet_id", values="revenue_sgd", aggfunc="sum")


### 🛠️ Group Exercise 2 — Integration (12 min)

> **(a) Worked for you** — the cell below joins region onto the monthly table and totals revenue
> by region. Read it, run it.
>
> **(b) Fill in the blanks** — how much rent does each region pay per month?
> ```python
> outlets.groupby("______")["monthly_rent_sgd"].sum()
> ```
> *Expected:* Central pays the most — three of the five outlets are Central.
>
> **(c) From scratch** — using `staffed` from 2.2, find the average `rev_per_staff_hour` for each
> outlet across the whole period. Which outlet earns the least per staff hour?
> *Expected:* five rows, and `OUT-03` is clearly the worst of the four permanent outlets.
>
> **(d) Decide and justify** — you have to report chain revenue to the owner. Do you use the
> `inner` join (which drops the pop-up kiosk) or the `left` join (which keeps it with a blank
> name)? There is a defensible answer either way — say which and why in one sentence.


**(a) Worked example** — revenue by region:


In [ ]:
# 👉 Join to get `region`, then group by it. Rows whose outlet_id is missing from `outlets`
#    get NaN for region -- so `dropna=False` keeps them visible instead of hiding them.
(
    monthly.merge(outlets[["outlet_id", "region"]], on="outlet_id", how="left")
    .groupby("region", dropna=False)["revenue_sgd"]
    .sum()
    .round(0)
)


---

# ☕ Break — 10 minutes

**Where we are:** the tables are joined and reshaped. Every row now knows its outlet name, region,
rent and staffing, and the targets are joinable.

**Next up:** Part 3 — the summarising engine. `groupby`, `pivot_table` and `crosstab`, and the two
tables that decide the lease.


## Part 3: Aggregation & Reporting

**Learning outcome 3:** *Aggregate data using `groupby`, pivot tables and cross-tabulations to
generate summary reports.*

**Goal:** turn 6,840 rows into the four or five numbers that a decision actually needs.

⏱️ ~45 min including Group Exercise 3


### 3.1: `groupby` — split, apply, combine

Three steps, always in this order:

1. **Split** the rows into groups by some key.
2. **Apply** a calculation to each group, independently.
3. **Combine** the answers into one table.


In [ ]:
# 👉 A drill you can check by hand: six rows, two groups.
drill = pd.DataFrame({
    "outlet": ["A", "A", "A", "B", "B", "B"],
    "daypart": ["Morning", "Midday", "Evening"] * 2,
    "revenue": [100, 60, 40, 30, 70, 90],
})

drill


In [ ]:
# 👉 Split by outlet, apply sum, combine. A: 100+60+40 = 200. B: 30+70+90 = 190.
drill.groupby("outlet")["revenue"].sum()


In [ ]:
# 👉 Group by TWO keys and you get one row per combination -- a MultiIndex (hierarchical index).
drill.groupby(["outlet", "daypart"])["revenue"].sum()


In [ ]:
# 👉 `.unstack()` lifts the last index level up into columns. This is exactly a pivot table,
#    built from a groupby. Same numbers, more readable shape.
drill.groupby(["outlet", "daypart"])["revenue"].sum().unstack()


In [ ]:
# 👉 Now the real thing. Total revenue per outlet across 18 months.
sales.groupby("outlet_id")["revenue_sgd"].sum().round(0)


### 3.2: `.agg()` — several questions in one pass

`.agg()` applies more than one function at a time, and lets you name the output columns. One pass
over the data, one readable table.


In [ ]:
# 👉 Named aggregation: each argument is `new_column=("source_column", "function")`.
#    Read it as a specification of the report you want.
outlet_report = sales.groupby("outlet_id").agg(
    revenue=("revenue_sgd", "sum"),
    tickets=("tickets", "sum"),
    trading_days=("date", "nunique"),
    best_day=("revenue_sgd", "max"),
)

# 👉 Derived columns come after the aggregation, from the aggregated numbers.
outlet_report["avg_ticket"] = (outlet_report["revenue"] / outlet_report["tickets"]).round(2)
outlet_report["revenue_per_day"] = (outlet_report["revenue"] / outlet_report["trading_days"]).round(0)
outlet_report["revenue"] = outlet_report["revenue"].round(0)
outlet_report["best_day"] = outlet_report["best_day"].round(0)

outlet_report


> **`nunique` vs `count` vs `size`.** `count` counts non-null values, `size` counts rows including
> nulls, and `nunique` counts distinct values. `trading_days` above had to be `nunique` — there are
> three rows per day, so `count` would have said 1,641 trading days in an 18-month period.


### 3.3: `pivot_table` — the two-dimensional summary

A pivot table is a `groupby` on two keys with the second one spread across the top. Four decisions:
**index** (down the side), **columns** (across the top), **values** (the number), **aggfunc** (how
to squash it).


In [ ]:
# 👉 Outlet down the side, daypart across the top, revenue in the middle, added up.
#    `margins=True` adds the "All" row and column -- the totals, and a free beat-4 check.
daypart_mix = sales.pivot_table(
    index="outlet_id",
    columns="daypart",
    values="revenue_sgd",
    aggfunc="sum",
    margins=True,
).round(0)

daypart_mix


In [ ]:
# 👉 Absolute dollars hide the pattern because the outlets are different sizes. Convert each row
#    to percentages of its own total: `div` divides, `axis=0` means "row by row".
mix_pct = (
    daypart_mix.drop(index="All").drop(columns="All")
    .div(daypart_mix.drop(index="All")["All"], axis=0)
    .mul(100).round(1)
)

mix_pct[["Morning", "Midday", "Evening"]]


> **Read the Morning column.** OUT-01 and OUT-03 make about half their money before 11am — office
> workers on the way in. OUT-04 makes barely a third in the morning and a third in the evening —
> that is a neighbourhood, not a commute. Same chain, two different businesses, and the marketing
> that works on one will not work on the other.


In [ ]:
# 👉 pivot_table takes several aggfuncs at once. Note what happens to the column headers:
#    they become two levels deep (aggfunc, then daypart).
sales.pivot_table(
    index="outlet_id", columns="daypart", values="revenue_sgd", aggfunc=["mean", "count"]
).round(0)


### 3.4: `crosstab` — counting combinations

`crosstab` is a pivot table specialised for **frequency**: how often does each combination occur?
This needs one row per event, so we switch to the ticket-level file for one week.


In [ ]:
# 👉 One row per till receipt, for the week of 16-22 June 2025.
tickets = pd.read_csv("../data/tickets_week.csv", parse_dates=["txn_datetime"])

tickets.head(3)


In [ ]:
# 👉 How many receipts for each payment method, in each part of the day? Plain counts.
pd.crosstab(tickets["payment_method"], tickets["daypart"])


In [ ]:
# 👉 Counts are hard to compare between columns of different sizes. `normalize="columns"`
#    turns each column into proportions of itself, so the columns are comparable.
(pd.crosstab(tickets["payment_method"], tickets["daypart"], normalize="columns") * 100).round(1)


In [ ]:
# 👉 crosstab can aggregate a value instead of counting, with `values=` + `aggfunc=`.
#    Average ticket size by outlet and category -- who is buying the expensive things?
pd.crosstab(
    tickets["outlet_id"], tickets["category"], values=tickets["amount_sgd"], aggfunc="mean"
).round(2)


### 3.5: Correlation — do two columns move together?

`.corr()` gives a number between -1 and 1 and, unlike covariance, it does not depend on the units.


In [ ]:
# 👉 Does staffing track revenue? Use the weekly table we merged in 2.2.
print("correlation, revenue vs staff hours:", staffed["revenue_sgd"].corr(staffed["staff_hours"]).round(3))

# 👉 Covariance answers the same "do they move together" question, but its size depends on the
#    units, so on its own it is uninterpretable. Correlation is covariance, normalised.
print("covariance (same relationship, unreadable scale):", round(staffed["revenue_sgd"].cov(staffed["staff_hours"]), 1))


In [ ]:
# 👉 On a DataFrame, `.corr()` gives every pair at once. Here: do the outlets' monthly revenues
#    move together? Compare the OUT-03 / OUT-04 cell with the others.
wide_view.corr().round(2)


> **Careful.** OUT-03 and OUT-04 are strongly *negatively* correlated, and nothing OUT-04 does
> takes money from OUT-03 — they are 8km apart with different customers. One is declining and the
> other is growing over the same 18 months, and correlation cannot tell that apart from a cause.
> Use it to *find candidates to investigate*, never as the finding itself.
>
> And look at the OUT-05 row: the kiosk traded for **three months**, so every number in it comes
> from three data points. A correlation computed from three points is a coincidence with a decimal
> place on it. Always ask how many observations are behind a correlation before you quote it.


In [ ]:
# 👉 The staffing question, properly. Revenue per staff hour, per outlet, for the last two quarters.
recent = staffed[staffed["week_start"] >= "2025-01-01"]

staffing = recent.groupby("outlet_id").agg(
    revenue=("revenue_sgd", "sum"),
    staff_hours=("staff_hours", "sum"),
)
staffing["rev_per_staff_hour"] = (staffing["revenue"] / staffing["staff_hours"]).round(2)
staffing["revenue"] = staffing["revenue"].round(0)
staffing["staff_hours"] = staffing["staff_hours"].round(0)

staffing.sort_values("rev_per_staff_hour")


> **Tick off finding #3.** Marina Bay earns the fewest dollars per staff hour of the four real
> outlets -- about \$20 an hour against \$25--27. Its revenue fell in November; its roster did not. The rota is still sized for the café
> it used to be — and that is a fixable problem worth real money, quite separate from the lease.


### 🛠️ Group Exercise 3 — Aggregation (10 min)

Split (a), (b), (c) across your group, then compare answers.

> **(a) Fill in the blanks** — average revenue per day by weekend vs weekday, per outlet:
> ```python
> sales.pivot_table(index="outlet_id", columns="__________", values="revenue_sgd", aggfunc="____")
> ```
> *Expected:* a 5 × 2 table. `OUT-01`'s weekend number is far below its weekday number; `OUT-02`'s
> is above.
>
> **(b) From scratch** — using `tickets`, build a crosstab of `category` against `daypart`
> normalised by column. Which category's share is biggest in the Evening?
>
> **(c) From scratch, two steps** — for 2025 only (filter with `sales["date"] >= "2025-01-01"`),
> build a table with one row per outlet showing total revenue and revenue **per seat**. You will
> need to merge in `seats` from `outlets`.
> *Hint:* filter → groupby → merge → divide. *Expected:* `OUT-01` earns the most per seat; the
> kiosk has no seats recorded, so it comes out as `NaN` — which is the honest answer, not a bug.
>
> **(d) Explain, no code** — `pivot_table` defaults to `aggfunc="mean"`, and `crosstab` defaults
> to counting. If you forgot that and used `pivot_table` to "count receipts by payment method",
> what would you actually get, and would it look wrong?


---

## Part 4: From Table to Decision

**Learning outcome 4:** *Apply the split-apply-combine pattern to answer complex analytical
questions on real datasets.*

**Goal:** the owner does not want the notebook. She wants one table she can act on. Everything
above was practice for these two cells.

⏱️ ~20 min


In [ ]:
# 👉 Beat 1: the question is "which outlets are getting better or worse, and by how much?"
#    Beat 2: the grain is one row per outlet. Two windows: the two most recent quarters
#    (2025 H1) against the same period a year earlier -- like-for-like, no seasonality excuse.
h1_2025 = sales[(sales["date"] >= "2025-01-01") & (sales["date"] <= "2025-06-30")]
h1_2024 = sales[(sales["date"] >= "2024-01-01") & (sales["date"] <= "2024-06-30")]

summary = pd.DataFrame({
    "rev_2025_h1": h1_2025.groupby("outlet_id")["revenue_sgd"].sum(),
    "rev_2024_h1": h1_2024.groupby("outlet_id")["revenue_sgd"].sum(),
})

# 👉 Beat 3: change is a percentage here, because the outlets are different sizes.
summary["change_pct"] = ((summary["rev_2025_h1"] / summary["rev_2024_h1"] - 1) * 100).round(1)

summary.round(0)


In [ ]:
# 👉 Now attach what it costs to run each one, and what each dollar of rent buys.
decision = (
    summary.merge(outlets.set_index("outlet_id")[["outlet_name", "monthly_rent_sgd", "seats"]],
                  left_index=True, right_index=True, how="left")
    .merge(staffing[["rev_per_staff_hour"]], left_index=True, right_index=True, how="left")
)

# 👉 Six months of rent against six months of revenue: rent as a share of takings.
decision["rent_pct_of_revenue"] = (
    decision["monthly_rent_sgd"] * 6 / decision["rev_2025_h1"] * 100
).round(1)

decision = decision[[
    "outlet_name", "rev_2025_h1", "change_pct", "rent_pct_of_revenue", "rev_per_staff_hour"
]].sort_values("change_pct")

decision.round(1)


**The answer, in three sentences.** Chain revenue is flat because Holland Village's growth is
almost exactly cancelling Marina Bay's decline. Marina Bay's fall is a step dated to the first week
of November 2024, not a drift — and it is now paying the highest rent as a share of takings, while
earning the least per staff hour. The lease decision is therefore not "is the chain healthy?" but
"can Marina Bay's November step be reversed, and if not, is that rent still worth paying?"

Notice what the table does **not** do: it does not say "close Marina Bay". It gives the owner the
three numbers that make her decision, and it makes them comparable.


In [ ]:
# 👉 Save the summary tables for Lesson 1.10, where they become the owner's one slide.
decision.to_csv("../data/lesson19_decision.csv")
monthly_by_outlet.to_csv("../data/lesson19_monthly_by_outlet.csv")

print("saved:")
print("  data/lesson19_decision.csv          <- the decision table")
print("  data/lesson19_monthly_by_outlet.csv <- monthly revenue by outlet")


## 🎯 Wrap-Up

1. **A date is a type.** Nothing about time works until `pd.to_datetime` has run.
2. **`resample` changes the grain of time; `rolling` smooths it.** Use `rolling` to tell a step
   (something happened on a date) from a slope (something is slowly changing).
3. **The `how` of a join is a business decision.** `inner` silently deleted $61,310. Default to
   `left`, then check for nulls with `indicator=True`.
4. **Long format is for computers, wide format is for people.** `melt` before you join; `pivot`
   before you show.
5. **Split-apply-combine is the engine.** `groupby` → `.agg()` → `pivot_table` → `crosstab` are
   four faces of one idea.
6. **Always run beat 4.** Grouped totals must tie back to ungrouped totals, or something was lost.
7. **An aggregate hides as much as it reveals.** A flat chain total was two opposite trends. The
   fix is not more data — it is a finer grain.

**Next Steps:**
- Complete the [Assignment](./assignment.md) — the Q3 review pack.
- Next lesson: **1.10 Data Visualisation & Storytelling** — the owner gets 20 seconds and one
  slide. Your `decision` table has to survive the trip.


### 📦 Appendix — self-study

These are in `reference.md` with worked examples, and are not taught live:

- **Hierarchical (Multi-)indexes:** `set_index` with two columns, `swaplevel`, `sort_index`,
  `.xs()` cross-sections.
- **`concat` vs `merge`:** stacking tables that share a shape, versus joining on a key.
- **`join` vs `merge`:** `join` works on the index by default; `merge` works on columns.
- **Time zones and business days:** `tz_localize`, `tz_convert`, `bdate_range`, custom offsets.
- **`.resample().agg()` and `.rolling().agg()`:** several statistics per bucket or window.
- **`stack` / `unstack`:** the index-level equivalents of `melt` / `pivot`.
